Import Required Libraries

In [ ]:
import os
import shutil
import zipfile
import tarfile
import gzip
import bz2
import lzma
from pathlib import Path
import tempfile

Core Extraction Functions

In [18]:
def is_archive(file_path):
    """
    Check if a file is a supported archive format.
    
    Args:
        file_path: Path to the file to check
        
    Returns:
        Boolean indicating if file is an archive
    """
    archive_extensions = [
        '.zip', '.tar', '.gz', '.bz2', '.xz', 
        '.tgz', '.tbz2', '.txz', '.tar.gz', 
        '.tar.bz2', '.tar.xz', '.rar', '.7z'
    ]
    
    file_lower = str(file_path).lower()
    return any(file_lower.endswith(ext) for ext in archive_extensions)


def extract_archive(archive_path, extract_to):
    """
    Extract an archive file to a specified directory.
    
    Args:
        archive_path: Path to the archive file
        extract_to: Directory where contents should be extracted
        
    Returns:
        Boolean indicating success
    """
    archive_path = Path(archive_path)
    extract_to = Path(extract_to)
    
    try:
        # Create extraction directory
        extract_to.mkdir(parents=True, exist_ok=True)
        
        # ZIP files
        if archive_path.suffix.lower() == '.zip':
            with zipfile.ZipFile(archive_path, 'r') as zip_ref:
                zip_ref.extractall(extract_to)
            return True
        
        # TAR files (including .tar.gz, .tar.bz2, .tar.xz)
        elif '.tar' in archive_path.suffixes or archive_path.suffix in ['.tgz', '.tbz2', '.txz']:
            with tarfile.open(archive_path, 'r:*') as tar_ref:
                tar_ref.extractall(extract_to)
            return True
        
        # GZIP files (single file compression)
        elif archive_path.suffix.lower() == '.gz' and '.tar' not in str(archive_path):
            output_file = extract_to / archive_path.stem
            with gzip.open(archive_path, 'rb') as f_in:
                with open(output_file, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
            return True
        
        # BZIP2 files (single file compression)
        elif archive_path.suffix.lower() == '.bz2' and '.tar' not in str(archive_path):
            output_file = extract_to / archive_path.stem
            with bz2.open(archive_path, 'rb') as f_in:
                with open(output_file, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
            return True
        
        # XZ files (single file compression)
        elif archive_path.suffix.lower() == '.xz' and '.tar' not in str(archive_path):
            output_file = extract_to / archive_path.stem
            with lzma.open(archive_path, 'rb') as f_in:
                with open(output_file, 'wb') as f_out:
                    shutil.copyfileobj(f_in, f_out)
            return True
        
        else:
            print(f"  ⚠️  Unsupported archive format: {archive_path.name}")
            return False
            
    except Exception as e:
        print(f"  ❌ Error extracting {archive_path.name}: {str(e)}")
        return False


def get_archive_stem(archive_path):
    """
    Get the name without archive extensions (handles .tar.gz, .tar.bz2, etc.)
    
    Args:
        archive_path: Path to archive file
        
    Returns:
        String with the base name without archive extensions
    """
    archive_path = Path(archive_path)
    name = archive_path.name
    
    # Handle compound extensions like .tar.gz
    compound_extensions = ['.tar.gz', '.tar.bz2', '.tar.xz']
    for ext in compound_extensions:
        if name.lower().endswith(ext):
            return name[:-len(ext)]
    
    # Handle simple extensions
    return archive_path.stem


def process_directory(source_dir, output_dir, current_depth=0, max_depth=10):
    """
    Recursively process a directory: extract archives and copy regular files.
    
    Args:
        source_dir: Source directory to process
        output_dir: Output directory where results should go
        current_depth: Current recursion depth
        max_depth: Maximum recursion depth
    """
    source_dir = Path(source_dir)
    output_dir = Path(output_dir)
    
    # Prevent infinite recursion
    if current_depth > max_depth:
        print(f"  ⚠️  Maximum recursion depth ({max_depth}) reached. Stopping.")
        return
    
    # Create output directory
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Process all items in source directory
    for item in sorted(source_dir.iterdir()):
        indent = "  " * current_depth
        
        if item.is_file():
            if is_archive(item):
                # Extract archive to a folder with the same name (without extension)
                folder_name = get_archive_stem(item)
                extract_folder = output_dir / folder_name
                
                print(f"{indent}📦 Extracting: {item.name} → {folder_name}/")
                
                # Extract the archive
                if extract_archive(item, extract_folder):
                    # Recursively process the extracted contents
                    process_directory(extract_folder, extract_folder, current_depth + 1, max_depth)
            else:
                # Copy regular files
                output_file = output_dir / item.name
                print(f"{indent}📄 Copying: {item.name}")
                shutil.copy2(item, output_file)
        
        elif item.is_dir():
            # Process subdirectories
            print(f"{indent}📁 Processing directory: {item.name}/")
            new_output_dir = output_dir / item.name
            process_directory(item, new_output_dir, current_depth + 1, max_depth)


Main Extraction Function

In [19]:
def extract_all_archives(input_path, output_dir=None):
    """
    Main function to extract all archives recursively to a new directory.
    
    Args:
        input_path: Path to input directory or archive file
        output_dir: Output directory (if None, creates one with '_extracted' suffix)
    """
    input_path = Path(input_path).resolve()
    
    # Validate input
    if not input_path.exists():
        raise FileNotFoundError(f"Input path does not exist: {input_path}")
    
    # Determine output directory
    if output_dir is None:
        if input_path.is_file():
            # For a file, create output directory next to it
            output_dir = input_path.parent / f"{get_archive_stem(input_path)}_extracted"
        else:
            # For a directory, create output directory next to it
            output_dir = input_path.parent / f"{input_path.name}_extracted"
    else:
        output_dir = Path(output_dir).resolve()
    
    # Create output directory
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("=" * 70)
    print("🚀 RECURSIVE ARCHIVE EXTRACTOR")
    print("=" * 70)
    print(f"📂 Input:  {input_path}")
    print(f"📂 Output: {output_dir}")
    print("=" * 70)
    
    # Process based on input type
    if input_path.is_file() and is_archive(input_path):
        # Extract the main archive
        folder_name = get_archive_stem(input_path)
        extract_folder = output_dir / folder_name
        
        print(f"📦 Extracting main archive: {input_path.name} → {folder_name}/")
        
        if extract_archive(input_path, extract_folder):
            # Recursively process extracted contents
            process_directory(extract_folder, extract_folder, 1)
    
    elif input_path.is_dir():
        # Process the entire directory
        process_directory(input_path, output_dir)
    
    else:
        print(f"❌ Input is not a valid archive or directory: {input_path}")
        return
    
    print("=" * 70)
    print(f"✅ Extraction complete!")
    print(f"📂 Results saved to: {output_dir}")
    print("=" * 70)

Extract from a zip file

In [ ]:
extract_all_archives("/Users/apple/Downloads/archive.zip")

Extract from a directory containing archives

In [ ]:
extract_all_archives("/Users/apple/Downloads/Sword_IT_Data_Package")

Starting in-place recursive extraction...
Working directory: /Users/apple/Downloads/Sword_IT_Data_Package
------------------------------------------------------------
Processing directory: OGA_Batch/
  Extracting: UK_10.zip -> UK_10/
    Processing directory: UK_10/
      Processing directory: 10_01-A6/
        Processing directory: WELL_PATH/
          Processing directory: WDD_QC/
        Processing directory: LOGS/
          Processing directory: JWL_DATACO/
      Processing directory: 10_01-A8/
        Processing directory: LOGS/
          Processing directory: JWL_DATACO/
      Processing directory: 10_01-A13/
        Processing directory: WELL_PATH/
          Processing directory: WDD_QC/
        Processing directory: LOGS/
          Processing directory: JWL_DATACO/
        Processing directory: PETROPHYSICS/
          Processing directory: CPI_DIGITISED/
      Processing directory: 10_01-A22/
        Processing directory: WELL_PATH/
          Processing directory: WDD_QC/
     